In [1]:
!pip install -U transformers accelerate datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATA_PATH = "/content/drive/MyDrive/KPDL/sampled_dataset_labeled_audited.csv"

In [4]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, precision_score, recall_score,classification_report, confusion_matrix,precision_recall_curve

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng: {device}")

Đang sử dụng: cuda


In [5]:
df = pd.read_csv(DATA_PATH)

print("COLUMNS:", df.columns)
df.head()

COLUMNS: Index(['rating', 'title', 'text', 'verified_purchase', 'quality_complaint'], dtype='object')


,rating,title,text,verified_purchase,quality_complaint
0,5,works great,product works great,True,0
1,3,three stars,don't pair with echo voice control remote ok,True,0
2,3,works great in the hallway,works great in the hallway but my outlets requ...,True,0
3,3,three stars,they work but the shipping was really slow,True,0
4,2,only one card out of a 2 pack is reliable,bought a two pack 64gb each put them both in m...,True,0


In [6]:
df.columns = df.columns.str.strip().str.lower()

if "label" not in df.columns:
    if "quality_complaint" in df.columns:
        df["label"] = df["quality_complaint"]
    else:
        raise Exception(" Không tìm thấy cột label")

print(df["label"].value_counts())

label
0    3535
1    1465
Name: count, dtype: int64


In [7]:
df["model_text"] = df["title"].fillna("") + " " + df["text"].fillna("")

X = df["model_text"]
y = df["label"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(len(X_train), len(X_val), len(X_test))

3500 750 750


In [8]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=3,
    max_df=0.95,
    max_features=100000,
    sublinear_tf=True
)

X_train_vec = tfidf.fit_transform(X_train)
X_val_vec = tfidf.transform(X_val)
X_test_vec = tfidf.transform(X_test)

In [9]:
lr = LogisticRegression(class_weight="balanced", max_iter=1000)
lr.fit(X_train_vec, y_train)

probs_lr = lr.predict_proba(X_val_vec)[:,1]

In [10]:
svm = LinearSVC(class_weight="balanced")
svm.fit(X_train_vec, y_train)

svm_calibrated = CalibratedClassifierCV(svm, method="sigmoid")
svm_calibrated.fit(X_train_vec, y_train)

probs_svm = svm_calibrated.predict_proba(X_val_vec)[:,1]

In [11]:
def find_best_threshold(y_true, probs):
    best_th, best_f1 = 0, 0

    for th in np.arange(0.3, 0.71, 0.05):
        preds = (probs >= th).astype(int)
        f1 = f1_score(y_true, preds)

        if f1 > best_f1:
            best_f1 = f1
            best_th = th

    return best_th, best_f1

th_lr, f1_lr = find_best_threshold(y_val, probs_lr)
th_svm, f1_svm = find_best_threshold(y_val, probs_svm)

print("LR:", th_lr, f1_lr)
print("SVM:", th_svm, f1_svm)

LR: 0.49999999999999994 0.7625
SVM: 0.39999999999999997 0.7868131868131868


In [12]:
def evaluate(y_true, probs, th):
    preds = (probs >= th).astype(int)

    return {
        "F1": f1_score(y_true, preds),
        "Precision": precision_score(y_true, preds),
        "Recall": recall_score(y_true, preds)
    }

res_lr = evaluate(y_val, probs_lr, th_lr)
res_svm = evaluate(y_val, probs_svm, th_svm)

print("Logistic:", res_lr)
print("SVM:", res_svm)

Logistic: {'F1': 0.7625, 'Precision': 0.7038461538461539, 'Recall': 0.8318181818181818}
SVM: {'F1': 0.7868131868131868, 'Precision': 0.7617021276595745, 'Recall': 0.8136363636363636}


In [13]:
df_val_analysis = pd.DataFrame({
    "text": X_val,
    "label": y_val,
    "prob": probs_svm
})

df_val_analysis["pred"] = (df_val_analysis["prob"] >= th_svm).astype(int)

print("False Negative:")
print(df_val_analysis[(df_val_analysis.label==1) & (df_val_analysis.pred==0)].head())

print("False Positive:")
print(df_val_analysis[(df_val_analysis.label==0) & (df_val_analysis.pred==1)].head())

False Negative:
                                                   text  label      prob  pred
4601  serious power issues beware this laptop came t...      1  0.190370     0
1330  one star for the price it jammed up on me firs...      1  0.107679     0
325   doesn t work i gave this as a gift so the pers...      1  0.263926     0
4106  i went online and looked at the youtube videos...      1  0.373537     0
3915  like the inside of a jewelry box just not at a...      1  0.059030     0
False Positive:
                                                   text  label      prob  pred
1616  motion detection is bad these things pretty mu...      0  0.553461     1
370   warning do not order from this company my orde...      0  0.890410     1
2972  great idea but a few of the challenges i have ...      0  0.429995     1
1436  two stars not sure yet if i like it got use to...      0  0.434785     1
2161  you get what you pay for i guess gets the job ...      0  0.432926     1


In [14]:
if res_svm["F1"] >= res_lr["F1"]:
    final_model = svm_calibrated
    final_threshold = th_svm
    print("FINAL MODEL: SVM")
else:
    final_model = lr
    final_threshold = th_lr
    print("FINAL MODEL: LOGISTIC")

FINAL MODEL: SVM


In [15]:
probs_test = final_model.predict_proba(X_test_vec)[:,1]

res_test = evaluate(y_test, probs_test, final_threshold)

print("TEST RESULT:", res_test)

TEST RESULT: {'F1': 0.7427293064876958, 'Precision': 0.7280701754385965, 'Recall': 0.7579908675799086}


In [16]:
from datasets import Dataset

train_df = pd.DataFrame({
    "text": X_train.values,
    "label": y_train.values
})

val_df = pd.DataFrame({
    "text": X_val.values,
    "label": y_val.values
})

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

In [17]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids","attention_mask","label"])
val_ds.set_format(type="torch", columns=["input_ids","attention_mask","label"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [18]:
from transformers import DistilBertForSequenceClassification

model_bert = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    logging_steps=50
)

In [20]:
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

In [21]:
trainer.train()

Step,Training Loss
50,0.582361
100,0.407236
150,0.396687
200,0.353420
250,0.270516
300,0.223061
350,0.227327
400,0.233137


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=438, training_loss=0.3242565823472254, metrics={'train_runtime': 165.1816, 'train_samples_per_second': 42.378, 'train_steps_per_second': 2.652, 'total_flos': 463635895296000.0, 'train_loss': 0.3242565823472254, 'epoch': 2.0})

In [22]:
pred = trainer.predict(val_ds)

# logits → probability
probs_bert = pred.predictions[:,1]

In [23]:
th_bert, f1_bert = find_best_threshold(y_val, probs_bert)

res_bert = evaluate(y_val, probs_bert, th_bert)

print("BERT:", res_bert)

BERT: {'F1': 0.8695652173913043, 'Precision': 0.8755760368663594, 'Recall': 0.8636363636363636}


In [24]:
print("\nMODEL COMPARISON")
print("Logistic:", res_lr)
print("SVM:", res_svm)
print("BERT:", res_bert)


MODEL COMPARISON
Logistic: {'F1': 0.7625, 'Precision': 0.7038461538461539, 'Recall': 0.8318181818181818}
SVM: {'F1': 0.7868131868131868, 'Precision': 0.7617021276595745, 'Recall': 0.8136363636363636}
BERT: {'F1': 0.8695652173913043, 'Precision': 0.8755760368663594, 'Recall': 0.8636363636363636}
